In [1]:
"""
模型选择
过拟合和欠拟合
"""

# === 误差问题 ===
# 训练误差：模型在训练数据集之上的误差
# 泛化误差：模型在新数据集之上的误差

# === 数据问题 ===
# 通常面对的问题是，数据基数不够多的情况，解决方案为：K-则交叉验证
# 算法：将训练数据分割为K块，for i = 1,...,k,使用第i块作为验证数据集，其余作为训练数据集，报告k个验证集误差的平均

# === 阶段性总结 ===
# 训练数据集：训练模型参数
# 验证数据集：选择模型超参数
# 非大数据集上通常使用k-折交叉验证

# === 过拟合和欠拟合 ===
#           数据
#       简单        复杂
# 低    正常        欠拟合
# 高    过拟合      正常
# 模型的容量指的是拟合各种函数的能力，低容量的模型难以拟合所有的训练数据，高容量的模型可以记住所有的训练数据

# 给定一个模型的种类，将有两个主要因素
# 因素-1：参数个数
# 因素-2：参数值的选择范围

# VC维的用处：
#   提供了一个模型好的理论依据，它可以衡量训练误差和泛化误差之间的间隔

# === 阶段性总结 ===
# 模型容量需要匹配数据复杂度，否则会导致欠拟合和过拟合

'\n模型选择\n过拟合和欠拟合\n'

In [2]:
import math
import numpy as np
import torch
from torch import nn
from d2l import torch as d2l

In [3]:
max_degree = 20  # 多项式的最大阶数
n_train, n_test = 100, 100  # 训练和测试数据集大小
true_w = np.zeros(max_degree)  # 分配大量的空间
true_w[0:4] = np.array([5, 1.2, -3.4, 5.6])

features = np.random.normal(size=(n_train + n_test, 1))
np.random.shuffle(features)
poly_features = np.power(features, np.arange(max_degree).reshape(1, -1))
for i in range(max_degree):
    poly_features[:, i] /= math.gamma(i + 1)  # gamma(n)=(n-1)!
# labels的维度:(n_train+n_test,)
labels = np.dot(poly_features, true_w)
labels += np.random.normal(scale=0.1, size=labels.shape)

In [4]:
# NumPy ndarray转换为tensor
true_w, features, poly_features, labels = [torch.tensor(x, dtype=
    torch.float32) for x in [true_w, features, poly_features, labels]]

features[:2], poly_features[:2, :], labels[:2]

(tensor([[-1.3712],
         [ 0.6446]]),
 tensor([[ 1.0000e+00, -1.3712e+00,  9.4010e-01, -4.2969e-01,  1.4730e-01,
          -4.0395e-02,  9.2317e-03, -1.8084e-03,  3.0996e-04, -4.7224e-05,
           6.4753e-06, -8.0718e-07,  9.2235e-08, -9.7287e-09,  9.5286e-10,
          -8.7104e-11,  7.4649e-12, -6.0211e-13,  4.5868e-14, -3.3102e-15],
         [ 1.0000e+00,  6.4457e-01,  2.0774e-01,  4.4633e-02,  7.1923e-03,
           9.2720e-04,  9.9607e-05,  9.1720e-06,  7.3900e-07,  5.2926e-08,
           3.4115e-09,  1.9990e-10,  1.0738e-11,  5.3240e-13,  2.4512e-14,
           1.0533e-15,  4.2433e-17,  1.6089e-18,  5.7614e-20,  1.9545e-21]]),
 tensor([-2.3713,  5.4343]))

In [5]:
def evaluate_loss(net, data_iter, loss):  #@save
    """评估给定数据集上模型的损失"""
    metric = d2l.Accumulator(2)  # 损失的总和,样本数量
    for X, y in data_iter:
        out = net(X)
        y = y.reshape(out.shape)
        l = loss(out, y)
        metric.add(l.sum(), l.numel())
    return metric[0] / metric[1]

In [6]:
def train(train_features, test_features, train_labels, test_labels,
          num_epochs=400):
    loss = nn.MSELoss(reduction='none')
    input_shape = train_features.shape[-1]
    # 不设置偏置，因为我们已经在多项式中实现了它
    net = nn.Sequential(nn.Linear(input_shape, 1, bias=False))
    batch_size = min(10, train_labels.shape[0])
    train_iter = d2l.load_array((train_features, train_labels.reshape(-1,1)),
                                batch_size)
    test_iter = d2l.load_array((test_features, test_labels.reshape(-1,1)),
                               batch_size, is_train=False)
    trainer = torch.optim.SGD(net.parameters(), lr=0.01)
    animator = d2l.Animator(xlabel='epoch', ylabel='loss', yscale='log',
                            xlim=[1, num_epochs], ylim=[1e-3, 1e2],
                            legend=['train', 'test'])
    for epoch in range(num_epochs):
        d2l.train_epoch_ch3(net, train_iter, loss, trainer)
        if epoch == 0 or (epoch + 1) % 20 == 0:
            animator.add(epoch + 1, (evaluate_loss(net, train_iter, loss),
                                     evaluate_loss(net, test_iter, loss)))
    print('weight:', net[0].weight.data.numpy())

In [7]:
# # 从多项式特征中选择前4个维度，即1,x,x^2/2!,x^3/3!
# train(poly_features[:n_train, :4], poly_features[n_train:, :4],
#       labels[:n_train], labels[n_train:])

In [8]:
# # 从多项式特征中选择前2个维度，即1和x
# train(poly_features[:n_train, :2], poly_features[n_train:, :2],
#       labels[:n_train], labels[n_train:])

In [9]:
# # 从多项式特征中选取所有维度
# train(poly_features[:n_train, :], poly_features[n_train:, :],
#       labels[:n_train], labels[n_train:], num_epochs=1500)